In [ ]:
import pandas as pd
from transformers import AutoTokenizer,  AutoModelForSequenceClassification
from peft import PeftModel, PeftConfig
from tqdm import tqdm
import torch
import json
import argparse
import os
from openai import OpenAI
api_key = os.environ.get("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

In [52]:
parser = argparse.ArgumentParser(description='Rewardモデルの評価をするプログラム')
parser.add_argument('--eval_model', help='aifモデルのパス', default="../checkpoints/mlp/final_model")    # 必須の引数を追加
parser.add_argument("--eval_data_path", help="評価データのパス", default="../checkpoints/mlp/test.csv")
parser.add_argument("--output_csv_path", help="評価結果の出力先", default="../checkpoints/outputs/eval_reward_model.csv")
args = parser.parse_args([])
tqdm.pandas()
model_name = args.eval_model
hf_path = args.eval_data_path

In [47]:
# ベースモデルの読み込み
peft_config = PeftConfig.from_pretrained(args.eval_model)
model = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path=peft_config.base_model_name_or_path,
    torch_dtype=torch.bfloat16,
    num_labels=1,
    return_dict=True,
    )
model = PeftModel.from_pretrained(model, args.eval_model)
model.cuda()

tokenizer = AutoTokenizer.from_pretrained(model.config._name_or_path)
tokenizer.pad_token = tokenizer.eos_token
def evaluate(prompt):
    with torch.no_grad():
        encoded = tokenizer.encode(prompt, return_tensors="pt").cuda()
        logits = model(encoded).logits
    return logits[0][0].item()

df_eval = pd.read_csv(args.eval_data_path)
df_eval["aif"] = df_eval["text"].progress_apply(evaluate)
print("---正解ラベルの分布----")
print(df_eval["label"].value_counts())

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at llm-jp/llm-jp-3-1.8b-instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [80]:
def is_number(s: str) -> bool:
    try:
        float(s)
        return True
    except ValueError:
        return False
def gpt3(prompt):
    # OpenAI APIを使用してgpt-3.5-turboモデルで応答を評価する関数
    chat_completion = ""
    while not is_number(chat_completion[0]):
        chat_completion = client.chat.completions.create(
            model="gpt-3.5-turbo-0125",
            messages=[{"role": "user", "content": df_eval["text"][0]}],
            max_tokens=4
        ).choices[0].message.content
    return chat_completion[0]

In [74]:
df_eval["zeroshot"] = df_eval["text"].progress_map(gpt3)
df_eval["zeroshot"]

100%|██████████| 1358/1358 [14:38<00:00,  1.55it/s]


0       5
1       4
2       4
3       4
4       4
       ..
1353    4
1354    4
1355    4
1356    4
1357    4
Name: zeroshot, Length: 1358, dtype: object

In [75]:
df_eval["wsft"] = df_eval["text"].progress_apply(evaluate)
df_eval["wsft"]

100%|██████████| 1358/1358 [01:17<00:00, 17.57it/s]


0       4.78125
1       4.90625
2       5.06250
3       5.09375
4       5.09375
         ...   
1353    4.81250
1354    5.15625
1355    4.78125
1356    5.09375
1357    4.93750
Name: wsft, Length: 1358, dtype: float64

In [76]:
print(" ---相関係数の計算・w/SFT(spearman, pearson)---")
round(df_eval[["label", "wsft"]].corr("spearman").iloc[1, 0], 3), round(df_eval[["label", "wsft"]].corr("pearson").iloc[1, 0], 3)
print(" ---相関係数の計算・chatgpt(spearman, pearson)---")
round(df_eval[["label", "zeroshot"]].corr("spearman").iloc[1, 0], 3), round(df_eval[["label", "zeroshot"]].corr("pearson").iloc[1, 0], 3)
df_eval.to_csv(args.output_csv_path, index=False, encoding="utf-8-sig")

 ---相関係数の計算・w/SFT(spearman, pearson)---
 ---相関係数の計算・chatgpt(spearman, pearson)---


ValueError: could not convert string to float: ']'